# Exercise 1 — LangGraph Agents, Routing and Human Approval
**Lecture 6 · Slides 20–29 · Student exercise**

This notebook runs independently of the other two. Run setup first. Examples use **simulated data**; no emails, web searches or payments are performed.

## Setup
Set `OPENAI_API_KEY` and optionally `OPENAI_MODEL` / `OPENAI_ENDPOINT` in your environment or `.env`. For an Azure-specific endpoint, configure an appropriate Azure client instead.

In [ ]:
%pip install -q langgraph langchain-openai langchain-core python-dotenv pydantic tenacity

In [ ]:
import os
from dotenv import load_dotenv
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command

load_dotenv()

# Instructor-provided model settings. OPENAI_ENDPOINT is optional for a standard OpenAI endpoint.
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
model_args = {"model": model_name, "api_key": os.getenv("OPENAI_API_KEY")}
if os.getenv("OPENAI_ENDPOINT"):
    model_args["base_url"] = os.environ["OPENAI_ENDPOINT"]

assert model_args["api_key"], "Set OPENAI_API_KEY in .env (or your environment)."
llm = ChatOpenAI(**model_args)
print("Model configured:", model_name)

## Part A — State, tools, and agent routing (slides 20–23, 28)
**Core goal:** a tool-using agent should call tools when appropriate, process their output, and stop. Refer to the agent → tools → agent diagram.

In [ ]:
# Safe, simulated data: no network requests
ORDERS = {
    "12345": {"status": "delayed", "expected_delivery": "2026-10-18", "amount_eur": 150},
    "56789": {"status": "shipped", "expected_delivery": "2026-10-25", "amount_eur": 45},
}

@tool
def lookup_order(order_id: str) -> str:
    """Look up a simulated customer's order by order ID."""
    return str(ORDERS.get(order_id, "Order not found"))

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools_e1 = [lookup_order, add_numbers]
agent_model_e1 = llm.bind_tools(tools_e1)
print([t.name for t in tools_e1])

In [ ]:
def agent_e1(state: MessagesState):
    answer = agent_model_e1.invoke([
        SystemMessage(content="You are a concise customer-support assistant. Use tools when needed."),
        *state["messages"],
    ])
    return {"messages": [answer]}

def route_e1(state: MessagesState) -> Literal["tools", "end"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

### Your task: Complete graph entry and compile
Add the edge from `START` to `agent` (see slide 22). Then compile the graph. This is intentionally the only missing core line.

In [ ]:
builder_e1 = StateGraph(MessagesState)
builder_e1.add_node("agent", agent_e1)
builder_e1.add_node("tools", ToolNode(tools_e1))

# TODO: Replace the line below with: builder_e1.add_edge(START, "agent")
# builder_e1.add_edge(START, "agent")

builder_e1.add_conditional_edges("agent", route_e1, {"tools": "tools", "end": END})
builder_e1.add_edge("tools", "agent")

# Uncomment after completing the TODO:
# app_e1 = builder_e1.compile()

### Test the workflow
Uncomment the code after you finish the previous cell. Ask a second question that requires a tool.

In [ ]:
# result_e1 = app_e1.invoke({"messages": [HumanMessage(content="What is the status of order 12345?")]})
# for msg in result_e1["messages"]:
#     print(type(msg).__name__, "|", str(msg.content)[:200], "| tool calls:", getattr(msg, "tool_calls", []))

# TODO: Ask a question that requires BOTH the order lookup and arithmetic tool.

**Check your understanding:** Why is `tools → agent` necessary? Why does `agent_node` return only the *new* message?

## Part B — Transient error recovery (slide 24)
Try a deterministic simulated service that fails twice. Compare retry attempts with and without backoff.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

class TemporaryServiceError(Exception):
    pass

retry_calls = {"count": 0}

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=0.2, min=0.2, max=1), retry=retry_if_exception_type(TemporaryServiceError), reraise=True)
def simulated_status_service():
    retry_calls["count"] += 1
    print("Attempt", retry_calls["count"])
    if retry_calls["count"] < 3:
        raise TemporaryServiceError("Temporary outage")
    return "Service recovered"

print(simulated_status_service())

## Part C — Human approval (slides 25–27)
LangGraph suspends with `interrupt()`, saves state and accepts a human decision via `Command(resume=...)`. The two outcomes below are simulated.

In [ ]:
class RefundState(TypedDict):
    order_id: str
    amount_eur: float
    approved: bool | None
    outcome: str

def review_e1b(state: RefundState):
    approved = interrupt({
        "question": "Approve this simulated refund?",
        "order_id": state["order_id"],
        "amount_eur": state["amount_eur"],
    })
    return {"approved": bool(approved)}

def route_approval_e1b(state: RefundState) -> Literal["approve", "reject"]:
    return "approve" if state["approved"] is True else "reject"

def approve_e1b(state: RefundState):
    return {"outcome": f"SIMULATED refund approved for order {state['order_id']}"}

def reject_e1b(state: RefundState):
    return {"outcome": "No refund executed"}

builder_e1b = StateGraph(RefundState)
builder_e1b.add_node("review", review_e1b)
builder_e1b.add_node("approve", approve_e1b)
builder_e1b.add_node("reject", reject_e1b)
builder_e1b.add_edge(START, "review")
builder_e1b.add_conditional_edges("review", route_approval_e1b, {"approve": "approve", "reject": "reject"})
builder_e1b.add_edge("approve", END)
builder_e1b.add_edge("reject", END)
app_e1b = builder_e1b.compile(checkpointer=InMemorySaver())

In [ ]:
config_e1b = {"configurable": {"thread_id": "refund-demo-001"}}
paused_e1b = app_e1b.invoke(
    {"order_id": "12345", "amount_eur": 150.0, "approved": None, "outcome": ""},
    config_e1b,
)
print("Request to human:", paused_e1b["__interrupt__"])

# TODO: Try both True and False using FRESH thread IDs (rerun the previous cell
# with a different thread_id for a second approval scenario).
resumed_e1b = app_e1b.invoke(Command(resume=False), config_e1b)
print("Decision:", resumed_e1b["outcome"])

### ⭐ Optional challenges
1. Run the approval example twice using different thread IDs, approving one and rejecting the other.
2. Add a new safe tool (for instance `subtract_numbers`). Remember to rebind the new tool list and rebuild the graph.
3. Explain why actual payment calls should not run before an `interrupt()`.

**Success:** Students can recognize state, nodes, edges, tool calls, conditional routing and interrupt/resume.